# POLISY linkage: VRscores employers to Compustat firms, triangulated and audited

This notebook runs `polisy_triangulate.py`. Put that file in your **POLISY** folder in Google Drive first.
Everything the notebook writes goes to **POLISY/linkage** in Drive, so a disconnected session loses nothing.

**Order:** 1 setup and input check → A build the links → hand review (if needed) → B draw the audit → C audit summary →
D and E selection table and workforce panels → F the sensitivity models in Stata.

**The rule:** no political variable is read until step D. Do not run any regression with the new links before the audit
(step C) is finished. The instructions PDF explains every step.

In [ ]:
# 1. Setup: packages, Google Drive, the code, and a check of every input file
%pip install -q duckdb openpyxl
import os, sys, importlib
from pathlib import Path
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/POLISY")      # change this line if your POLISY folder is somewhere else
os.environ["POLISY_BASE"] = str(BASE)
os.environ.setdefault("POLISY_LAB_ROOT", str(BASE / "lab"))
sys.path.insert(0, str(BASE))
import polisy_triangulate as tri
importlib.reload(tri)

# If the check below says the Compustat file is MISSING, write its path here and run this cell again:
# tri.PATHS["compustat"] = BASE / "data" / "Compustat_Final.csv"
print("polisy_triangulate", tri.__version__)
inputs = tri.check_inputs()

## Step A: build the links

Builds every candidate pair, checks it against the independent sources, gives it a match type, forms the four link sets
and freezes the result. It reads names, years, worker counts, Compustat sizes, patents and DISCERN, and nothing political.

Read the printed lines `match types:` and `set ...`, then the three tables. If `review_list.xlsx` has rows, do the hand
review below and run this cell again with a new note.

In [ ]:
# A. Build the links, then show the tables
links = tri.build_links(note="first build")      # when you rebuild, say why: note="after review round 1"
OUT = BASE / "linkage"
display(pd.read_csv(OUT / "T1_match_types.csv"))
display(pd.read_csv(OUT / "T1b_link_sets.csv"))
display(pd.read_csv(OUT / "T3_agreement.csv"))

### Overview of every link (to read, filter and check by hand)

Writes **POLISY/linkage/all_links_overview.xlsx**: every candidate pair with the cleaned names and all the evidence
(sheet *All links*), one row per Compustat firm (*By firm*) and the summary tables. To record a decision, fill `accept`
in *All links* and save a copy in POLISY/linkage as `hand_decisions_<n>.xlsx`; then rerun step A.

In [ ]:
# Overview: every link in one workbook (reads only the frozen links, nothing political)
tri.export_overview()

### Hand review (only if review_list.xlsx has rows)

1. In Google Drive, open **POLISY/linkage/review_list.xlsx** (download it, or open it with Google Sheets).
2. For each row, fill **accept** (1 = same company or a subsidiary it owned in 2012-2021, 0 = not), **decided_by** and **note**.
   Leave accept empty when unsure. For a *conflict*, accept the right firm and reject the other.
3. Decide on names, sizes, company websites and 10-K subsidiary lists only. Never use party, donation, DIPI or VRscores share data.
4. Save it in **POLISY/linkage** as **hand_decisions.xlsx** (a later round: hand_decisions_2.xlsx, and so on).
   In Google Sheets: File > Download > Microsoft Excel, then upload it to POLISY/linkage.
5. Run step A again with `note="after review round 1"`.

## Step B: draw the audit (after the review is finished)

Draws 200 links at random within each match type (always the same 200, because the seed is fixed) and writes two
identical workbooks, **audit_coder1.xlsx** and **audit_coder2.xlsx**, for two people who code them separately.
**audit_key.csv** says which match type each row is: never give it to the coders.

In [ ]:
# B. Draw the audit sample (run once; a second run keeps the first sample)
key = tri.draw_audit()            # tri.draw_audit(n=100) is quicker, with wider intervals

## Step C: audit summary (after both coders have finished)

Upload both coded workbooks to **POLISY/linkage** under the same names, then run the cell. Rows on which the coders
disagree are listed in **audit_disagreements.csv**: agree on an answer together, write it in its `final` column (1, 0 or ?),
save it as **audit_resolution.csv** in POLISY/linkage and run the cell again.

In [ ]:
# C. Inter-coder agreement and precision by match type and by link set
T4 = tri.audit_summary()

## Steps D and E: selection table and workforce panels (only after step C)

These are the first steps that read political data (DIPI and the VRscores party counts). They refuse to run if
links_all.csv was changed after it was frozen.

In [ ]:
# D. Linked versus unlinked firms;  E. the firm-year workforce panel of each link set (CSV and Stata)
T2 = tri.selection_table()
display(T2.round(3))
tri.workforce_panels()

## Step F: the sensitivity models in Stata

Download from POLISY/linkage the four **workforce_*.dta** files and **linkage_sensitivity.do**. Edit the five lines marked
`EDIT` in the do-file (folder, your thesis file, the CEO score, the controls and the model of Table 5, Model 1), then run it.
It estimates the model once per link set and prints the four coefficients side by side.